# Module 17 — Week 6 — Bayesian Black-Box Optimisation Capstone

**W5 results: 3/8 improved (F5, F6, F8). F4 disaster — outlier threshold too aggressive.**

| Function | W5 Result | All-time Best | W6 Strategy |
|----------|-----------|--------------|-------------|
| F1 | 8.11e-05 | 2.82e-04 (W2) | Expand trust to r=0.15 |
| F2 | 0.583 | 0.611 (initial) | Tighten to r=0.10 — almost there |
| F3 | -0.0187 | -0.011 (W1) | Wider exploration — trust not working |
| F4 | -13.98 | -0.128 (W4) | Fix: raise outlier threshold to -2.0 |
| F5 | 2941.85 | 2941.85 (W5) | Tightest exploit on W5 peak |
| F6 | -0.2999 | -0.2999 (W5) | Trust on W5 best |
| F7 | 1.661 | 2.671 (W4) | Expand back to r=0.25 |
| F8 | 9.9275 | 9.9275 (W5) | Continue r=0.30 |

In [1]:
import matplotlib
matplotlib.use('Agg')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from sklearn.svm import SVC
from scipy.stats import norm
from scipy.stats.qmc import LatinHypercube
import warnings
import os
warnings.filterwarnings('ignore')

PLOTS_DIR = '/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/module-17/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

print('Module 17 — Week 6 — Bayesian Black-Box Optimisation')
print('All imports loaded successfully')

Module 17 — Week 6 — Bayesian Black-Box Optimisation
All imports loaded successfully


In [2]:
# ALL HISTORICAL SUBMISSIONS — W1 through W5

submitted_x_w1 = {
    1: [0.020584, 0.969910],
    2: [0.814691, 0.969505],
    3: [0.376075, 0.370839, 0.474761],
    4: [0.369789, 0.452786, 0.367951, 0.448446],
    5: [0.241041, 0.805036, 0.948951, 0.905090],
    6: [0.466959, 0.356875, 0.489683, 0.726384, 0.125125],
    7: [0.027698, 0.531762, 0.337094, 0.176133, 0.361503, 0.730849],
    8: [0.192432, 0.183093, 0.018724, 0.036362, 0.690267, 0.444236, 0.081374, 0.428967],
}
new_y_w1 = {
    1:  1.966e-321,
    2:  0.1292261555216582,
    3: -0.010707313301147062,
    4: -0.34595283782499875,
    5:  1450.9433021815964,
    6: -0.3611823990070205,
    7:  1.4058168801082682,
    8:  9.8915570907296,
}

submitted_x_w2 = {
    1: [0.591837, 0.591837],
    2: [0.000000, 1.000000],
    3: [0.421053, 1.000000, 1.000000],
    4: [0.909548, 0.568955, 0.762175, 0.811807],
    5: [0.204881, 0.877830, 0.879582, 0.870578],
    6: [0.851439, 0.906254, 0.506372, 0.594105, 0.708147],
    7: [0.097054, 0.432660, 0.338116, 0.122619, 0.296117, 0.886436],
    8: [0.076274, 0.101214, 0.383035, 0.338493, 0.113685, 0.882235, 0.615428, 0.796463],
}
new_y_w2 = {
    1:  0.00028209052469858225,
    2:  0.1709619176069506,
    3: -0.48304244384724265,
    4: -26.59459580774249,
    5:  1192.2995655092311,
    6: -1.9259411859252866,
    7:  1.2030170341293975,
    8:  9.0382459830856,
}

submitted_x_w3 = {
    1: [0.980000, 0.980000],
    2: [1.000000, 0.306122],
    3: [1.000000, 0.000000, 0.684211],
    4: [0.985601, 0.686679, 0.243615, 0.798556],
    5: [0.204881, 0.877830, 0.879582, 0.870578],
    6: [0.061416, 0.762464, 0.106527, 0.271402, 0.782742],
    7: [0.067189, 0.412831, 0.295130, 0.070570, 0.412599, 0.616173],
    8: [0.682757, 0.427203, 0.591529, 0.734064, 0.514947, 0.813984, 0.722156, 0.615073],
}
new_y_w3 = {
    1:  2.665897212344236e-174,
    2: -0.042550557700427774,
    3: -0.1840890683677661,
    4: -26.07041694623693,
    5:  1192.2995655092311,
    6: -2.508952125110497,
    7:  1.2533263563752521,
    8:  7.5792591902086,
}

submitted_x_w4 = {
    1: [0.278296, 0.020000],
    2: [0.685269, 0.947006],
    3: [0.403468, 0.441923, 0.497061],
    4: [0.352971, 0.651614, 0.805417, 0.616108],
    5: [0.167299, 0.881015, 0.978872, 0.954244],
    6: [0.334649, 0.293944, 0.500782, 0.769829, 0.074923],
    7: [0.206363, 0.281987, 0.389442, 0.281544, 0.218827, 0.711599],
    8: [0.095545, 0.327238, 0.051339, 0.269531, 0.555763, 0.417489, 0.285113, 0.613881],
}
new_y_w4 = {
    1: -2.6647756688938686e-133,
    2:  0.14705786268424045,
    3: -0.022992940111015336,
    4: -0.1283640964538999,
    5:  2496.347187728138,
    6: -0.38592078647528016,
    7:  2.6705394912160187,
    8:  9.8967959631939,
}

submitted_x_w5 = {
    1: [0.580092, 0.683225],
    2: [0.702813, 0.926626],
    3: [0.365086, 0.316421, 0.471038],
    4: [0.344700, 0.645505, 0.791987, 0.622463],
    5: [0.139557, 0.911522, 0.979905, 0.977049],
    6: [0.417831, 0.356959, 0.468069, 0.668531, 0.039515],
    7: [0.384097, 0.122113, 0.444891, 0.357064, 0.147383, 0.783086],
    8: [0.089787, 0.068251, 0.180968, 0.327284, 0.766207, 0.653365, 0.174832, 0.499246],
}
new_y_w5 = {
    1:  0.00008112997850454906,
    2:  0.5833602539566602,
    3: -0.018707796769607724,
    4: -13.979947691578896,
    5:  2941.854350298978,
    6: -0.29985775692426564,
    7:  1.660798293687705,
    8:  9.9275118625839,
}

print('All historical data loaded — W1 through W5')
print()
all_time_best = {
    1: (0.00028209052469858225, 'W2'),
    2: (0.611205,               'Initial'),
    3: (-0.010707313301147062,  'W1'),
    4: (-0.1283640964538999,    'W4'),
    5: (2941.854350298978,      'W5'),
    6: (-0.29985775692426564,   'W5'),
    7: (2.6705394912160187,     'W4'),
    8: (9.9275118625839,        'W5'),
}
print('All-time bests going into W6:')
for i in range(1,9):
    v, w = all_time_best[i]
    print(f'  F{i}: {v:.4e}  ({w})')

All historical data loaded — W1 through W5

All-time bests going into W6:
  F1: 2.8209e-04  (W2)
  F2: 6.1120e-01  (Initial)
  F3: -1.0707e-02  (W1)
  F4: -1.2836e-01  (W4)
  F5: 2.9419e+03  (W5)
  F6: -2.9986e-01  (W5)
  F7: 2.6705e+00  (W4)
  F8: 9.9275e+00  (W5)


In [3]:
base_path = '/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/data/'

descriptions = {
    1: 'Radiation Detection',
    2: 'Noisy ML Model',
    3: 'Drug Discovery',
    4: 'Warehouse Placement',
    5: 'Chemical Yield (STAR)',
    6: 'Cake Recipe',
    7: 'ML Hyperparameters',
    8: 'Complex 8D',
}

data = {}
for i in range(1, 9):
    X0 = np.load(f'{base_path}function_{i}/initial_inputs.npy')
    Y0 = np.load(f'{base_path}function_{i}/initial_outputs.npy')
    X_all = np.vstack([
        X0,
        np.array(submitted_x_w1[i]).reshape(1,-1),
        np.array(submitted_x_w2[i]).reshape(1,-1),
        np.array(submitted_x_w3[i]).reshape(1,-1),
        np.array(submitted_x_w4[i]).reshape(1,-1),
        np.array(submitted_x_w5[i]).reshape(1,-1),
    ])
    Y_all = np.concatenate([Y0,
        [new_y_w1[i]], [new_y_w2[i]], [new_y_w3[i]],
        [new_y_w4[i]], [new_y_w5[i]]])
    data[i] = {'X': X_all, 'Y': Y_all}

print(f'{"Fn":<4} {"Description":<24} {"N":<5} {"Dim":<5} {"All-time Best Y"}')
print('-' * 65)
for i in range(1,9):
    Y   = data[i]['Y']
    dim = data[i]['X'].shape[1]
    print(f'F{i:<3} {descriptions[i]:<24} {len(Y):<5} {dim:<5} {Y.max():.6e}')
print()
print('15 observations per function (10 initial + W1-W5)')

Fn   Description              N     Dim   All-time Best Y
-----------------------------------------------------------------
F1   Radiation Detection      15    2     2.820905e-04
F2   Noisy ML Model           15    2     6.112052e-01
F3   Drug Discovery           20    3     -1.070731e-02
F4   Warehouse Placement      35    4     -1.283641e-01
F5   Chemical Yield (STAR)    25    4     2.941854e+03
F6   Cake Recipe              25    5     -2.998578e-01
F7   ML Hyperparameters       35    6     2.670539e+00
F8   Complex 8D               45    8     9.927512e+00

15 observations per function (10 initial + W1-W5)


In [4]:
BOUND_LO, BOUND_HI = 0.02, 0.98

all_submitted_X = {
    i: [
        np.array(submitted_x_w1[i]),
        np.array(submitted_x_w2[i]),
        np.array(submitted_x_w3[i]),
        np.array(submitted_x_w4[i]),
        np.array(submitted_x_w5[i]),
    ]
    for i in range(1, 9)
}

def check_duplicate(next_x, func_num, threshold=0.015):
    return any(np.linalg.norm(next_x - x) < threshold for x in all_submitted_X[func_num])

def expected_improvement(mu, sigma, best_y_log, xi=0.01):
    imp = mu - best_y_log - xi
    Z   = imp / (sigma + 1e-9)
    ei  = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma < 1e-10] = 0.0
    return ei

def gp_predict_scalar(gp, x):
    return float(gp.predict(x.reshape(1,-1)).ravel()[0])

def build_search_grid(dim, trust_center=None, trust_radius=None, use_lhs=False, n=50000):
    N = n
    if trust_center is not None and trust_radius is not None:
        lo = np.clip(trust_center - trust_radius, BOUND_LO, BOUND_HI)
        hi = np.clip(trust_center + trust_radius, BOUND_LO, BOUND_HI)
        s  = LatinHypercube(d=dim, seed=42).random(n=N)
        return lo + s*(hi-lo), f'Trust-LHS r={trust_radius:.2f} {dim}D {N:,}pts'
    elif dim == 2:
        g = np.linspace(BOUND_LO, BOUND_HI, 224)
        XX, YY = np.meshgrid(g, g)
        return np.column_stack([XX.ravel(), YY.ravel()]), '224x224 grid ~50k pts'
    elif use_lhs:
        s = LatinHypercube(d=dim, seed=42).random(n=N)
        return BOUND_LO + s*(BOUND_HI-BOUND_LO), f'LHS {N:,}pts {dim}D'
    else:
        np.random.seed(42)
        return np.random.uniform(BOUND_LO, BOUND_HI, (N, dim)), f'Random {N:,}pts {dim}D'

def analyse_function_w6(func_num, beta_ucb=2.0, use_ei=True,
                         use_lhs=False, trust_center=None, trust_radius=None,
                         xi=0.01, length_scale=0.2,
                         remove_outliers=False, outlier_threshold=None):
    X, Y = data[func_num]['X'], data[func_num]['Y']
    dim      = X.shape[1]
    best_idx = np.argmax(Y)
    best_X   = X[best_idx] if trust_center is None else trust_center
    best_Y   = Y[best_idx]

    print(f'\n{"="*68}')
    print(f'F{func_num} {descriptions[func_num]} | Dim={dim} N={len(Y)} BestY={best_Y:.4e}')
    print(f'Best X* = [{" ".join(f"{v:.4f}" for v in X[best_idx])}]')
    print(f'W5 result = {new_y_w5[func_num]:.4e}')
    print('='*68)

    X_fit, Y_fit_raw = X, Y
    if remove_outliers and outlier_threshold is not None:
        mask = Y > outlier_threshold
        X_fit, Y_fit_raw = X[mask], Y[mask]
        print(f'  [Outlier] Removed {(~mask).sum()} pts (Y<{outlier_threshold}) — GP on {mask.sum()} pts')

    X_grid, grid_info = build_search_grid(
        dim,
        trust_center=best_X if trust_radius else None,
        trust_radius=trust_radius,
        use_lhs=use_lhs
    )
    print(f'  [Grid] {grid_info}')

    Y_log  = np.log(np.abs(Y_fit_raw)+1e-300)*np.sign(Y_fit_raw+1e-300)
    kernel = Matern(length_scale=length_scale, nu=2.5)
    gp     = GaussianProcessRegressor(kernel=kernel, alpha=1e-6,
                                       n_restarts_optimizer=3, normalize_y=True)
    gp.fit(X_fit, Y_log)

    mu_chk, std_chk = gp.predict(X[best_idx].reshape(1,-1), return_std=True)
    actual_log = np.log(np.abs(best_Y)+1e-300)*np.sign(best_Y+1e-300)
    print(f'  [GP] {gp.kernel_}')
    print(f'  [Sanity] pred={float(mu_chk.ravel()[0]):.4f} actual={actual_log:.4f} std={float(std_chk.ravel()[0]):.6f}')

    mu, sigma = gp.predict(X_grid, return_std=True)
    best_y_log = np.log(np.abs(best_Y)+1e-300)*np.sign(best_Y+1e-300)

    ucb   = mu + beta_ucb*sigma;  x_ucb = X_grid[np.argmax(ucb)]
    ei    = expected_improvement(mu, sigma, best_y_log, xi=xi)
    x_ei  = X_grid[np.argmax(ei)]

    print(f'  [UCB] b={beta_ucb} max={ucb.max():.4f} => [{" ".join(f"{v:.4f}" for v in x_ucb)}]')
    print(f'  [EI]  xi={xi} max={ei.max():.6f} => [{" ".join(f"{v:.4f}" for v in x_ei)}]')

    mu_u = gp_predict_scalar(gp, x_ucb)
    mu_e = gp_predict_scalar(gp, x_ei)
    next_x, winner = (x_ei,'EI') if (use_ei and mu_e >= mu_u) else (x_ucb,'UCB')
    print(f'  [Ensemble] UCB_mean={mu_u:.4f} EI_mean={mu_e:.4f} => winner={winner}')

    if check_duplicate(next_x, func_num):
        np.random.seed(99)
        next_x = np.clip(next_x + np.random.uniform(-0.03, 0.03, dim), BOUND_LO, BOUND_HI)
        print(f'  [Dup] Duplicate detected — point perturbed')

    print(f'  [Dist] Distance from best: {np.linalg.norm(next_x - X[best_idx]):.4f}')
    portal = '-'.join([f'{v:.6f}' for v in next_x])
    print(f'\n  >>> SUBMIT F{func_num}: {portal} <<<')
    return next_x, portal

print('Helpers ready — W6 version (tracks W1-W5 submissions)')

Helpers ready — W6 version (tracks W1-W5 submissions)


In [5]:
# F1 — Radiation Detection (2D)
# W5: 8.11e-05 — second best ever (W2 best is 2.82e-04)
# Trust region around W2 point is working — expand slightly to r=0.15
W2_BEST_X1 = np.array([0.591837, 0.591837])

next_x1, portal1 = analyse_function_w6(
    func_num     = 1,
    beta_ucb     = 1.0,
    use_ei       = True,
    use_lhs      = False,
    trust_center = W2_BEST_X1,
    trust_radius = 0.15,         # expanded from 0.10 — W5 point was at edge
    xi           = 0.001,
)


F1 Radiation Detection | Dim=2 N=15 BestY=2.8209e-04
Best X* = [0.5918 0.5918]
W5 result = 8.1130e-05
  [Grid] Trust-LHS r=0.15 2D 50,000pts
  [GP] Matern(length_scale=0.387, nu=2.5)
  [Sanity] pred=-8.1729 actual=-8.1733 std=0.235000
  [UCB] b=1.0 max=32.6639 => [0.4440 0.4422]
  [EI]  xi=0.001 max=16.049689 => [0.6534 0.6529]
  [Ensemble] UCB_mean=-40.4209 EI_mean=7.8731 => winner=EI
  [Dist] Distance from best: 0.0867

  >>> SUBMIT F1: 0.653384-0.652924 <<<


In [6]:
# F2 — Noisy ML Model (2D)
# W5: 0.583 — very close to initial best (0.611). Only 0.028 away.
# Tighten trust region to r=0.10 — we're right next to the peak
INIT_BEST_X2 = np.array([0.70263656, 0.9265642])

next_x2, portal2 = analyse_function_w6(
    func_num     = 2,
    beta_ucb     = 0.3,          # very exploitative — almost at the peak
    use_ei       = True,
    use_lhs      = False,
    trust_center = INIT_BEST_X2,
    trust_radius = 0.10,         # tighter than W5 (was 0.15)
    xi           = 0.005,
)


F2 Noisy ML Model | Dim=2 N=15 BestY=6.1121e-01
Best X* = [0.7026 0.9266]
W5 result = 5.8336e-01
  [Grid] Trust-LHS r=0.10 2D 50,000pts
  [GP] Matern(length_scale=0.011, nu=2.5)
  [Sanity] pred=-0.4924 actual=-0.4923 std=0.002207
  [UCB] b=0.3 max=0.8374 => [0.6945 0.9221]
  [EI]  xi=0.005 max=1.139729 => [0.6931 0.9203]
  [Ensemble] UCB_mean=0.4153 EI_mean=0.3084 => winner=UCB
  [Dup] Duplicate detected — point perturbed
  [Dist] Distance from best: 0.0056

  >>> SUBMIT F2: 0.704856-0.921380 <<<


In [7]:
# F3 — Drug Discovery (3D)
# W5: -0.0187 — trust region keeps missing W1 peak (-0.011)
# Try wider LHS exploration — current trust approach not working after 2 weeks
next_x3, portal3 = analyse_function_w6(
    func_num     = 3,
    beta_ucb     = 2.0,          # higher — explore more widely
    use_ei       = True,
    use_lhs      = True,         # wide LHS — no trust region
    trust_radius = None,
    xi           = 0.001,
)


F3 Drug Discovery | Dim=3 N=20 BestY=-1.0707e-02
Best X* = [0.3761 0.3708 0.4748]
W5 result = -1.8708e-02
  [Grid] LHS 50,000pts 3D
  [GP] Matern(length_scale=1e-05, nu=2.5)
  [Sanity] pred=4.5368 actual=4.5368 std=0.000926
  [UCB] b=2.0 max=4.4124 => [0.1517 0.8260 0.6228]
  [EI]  xi=0.001 max=0.005425 => [0.1517 0.8260 0.6228]
  [Ensemble] UCB_mean=2.5611 EI_mean=2.5611 => winner=EI
  [Dist] Distance from best: 0.5287

  >>> SUBMIT F3: 0.151659-0.826046-0.622768 <<<


In [8]:
# F4 — Warehouse Placement (4D)
# W5: -13.98 — DISASTER. Outlier threshold -5.0 left only 3 GP points.
# Fix: raise threshold to -2.0 so more good points are kept
# Also anchor trust to W4 best (-0.128) which is the all-time best
W4_BEST_X4 = np.array([0.352971, 0.651614, 0.805417, 0.616108])

next_x4, portal4 = analyse_function_w6(
    func_num          = 4,
    beta_ucb          = 1.5,
    use_ei            = True,
    use_lhs           = True,
    trust_center      = W4_BEST_X4,
    trust_radius      = 0.20,
    xi                = 0.01,
    remove_outliers   = True,
    outlier_threshold = -2.0,    # raised from -5.0 — keeps more valid points
)


F4 Warehouse Placement | Dim=4 N=35 BestY=-1.2836e-01
Best X* = [0.3530 0.6516 0.8054 0.6161]
W5 result = -1.3980e+01
  [Outlier] Removed 33 pts (Y<-2.0) — GP on 2 pts
  [Grid] Trust-LHS r=0.20 4D 50,000pts
  [GP] Matern(length_scale=1e-05, nu=2.5)
  [Sanity] pred=2.0529 actual=2.0529 std=0.000496
  [UCB] b=1.5 max=2.3007 => [0.2917 0.7148 0.9119 0.6645]
  [EI]  xi=0.01 max=0.039738 => [0.2917 0.7148 0.9119 0.6645]
  [Ensemble] UCB_mean=1.5572 EI_mean=1.5572 => winner=EI
  [Dist] Distance from best: 0.1464

  >>> SUBMIT F4: 0.291709-0.714786-0.911879-0.664486 <<<


In [9]:
# F5 — Chemical Yield (4D) — STAR PERFORMER
# W5: 2941.85 — new all-time best. Keep squeezing.
# Tightest trust around W5 best point
W5_BEST_X5 = np.array([0.139557, 0.911522, 0.979905, 0.977049])

next_x5, portal5 = analyse_function_w6(
    func_num     = 5,
    beta_ucb     = 0.05,
    use_ei       = True,
    use_lhs      = True,
    trust_center = W5_BEST_X5,
    trust_radius = 0.04,         # tighter than W5 (was 0.05)
    xi           = 0.01,
)


F5 Chemical Yield (STAR) | Dim=4 N=25 BestY=2.9419e+03
Best X* = [0.1396 0.9115 0.9799 0.9770]
W5 result = 2.9419e+03
  [Grid] Trust-LHS r=0.04 4D 50,000pts
  [GP] Matern(length_scale=0.431, nu=2.5)
  [Sanity] pred=7.9868 actual=7.9868 std=0.002259
  [UCB] b=0.05 max=8.0810 => [0.1024 0.9513 0.9777 0.9782]
  [EI]  xi=0.01 max=0.124474 => [0.1792 0.9511 0.9712 0.9796]
  [Ensemble] UCB_mean=8.0732 EI_mean=7.9718 => winner=UCB
  [Dist] Distance from best: 0.0545

  >>> SUBMIT F5: 0.102387-0.951309-0.977662-0.978193 <<<


In [10]:
# F6 — Cake Recipe (5D)
# W5: -0.2999 — first ever improvement! New all-time best.
# Trust region around W5 best point
W5_BEST_X6 = np.array([0.417831, 0.356959, 0.468069, 0.668531, 0.039515])

next_x6, portal6 = analyse_function_w6(
    func_num     = 6,
    beta_ucb     = 1.0,
    use_ei       = True,
    use_lhs      = True,
    trust_center = W5_BEST_X6,
    trust_radius = 0.15,
    xi           = 0.001,
)


F6 Cake Recipe | Dim=5 N=25 BestY=-2.9986e-01
Best X* = [0.4178 0.3570 0.4681 0.6685 0.0395]
W5 result = -2.9986e-01
  [Grid] Trust-LHS r=0.15 5D 50,000pts
  [GP] Matern(length_scale=0.625, nu=2.5)
  [Sanity] pred=1.2044 actual=1.2044 std=0.000564
  [UCB] b=1.0 max=1.3080 => [0.3442 0.4039 0.4277 0.5361 0.0201]
  [EI]  xi=0.001 max=0.043059 => [0.3944 0.3991 0.4111 0.5951 0.0206]
  [Ensemble] UCB_mean=1.1914 EI_mean=1.2327 => winner=EI
  [Dist] Distance from best: 0.1064

  >>> SUBMIT F6: 0.394360-0.399099-0.411100-0.595076-0.020599 <<<


In [11]:
# F7 — ML Hyperparameters (6D)
# W5: 1.661 — regression from W4 best (2.671). Trust radius was too tight.
# Expand back to r=0.25, anchor to W4 best point
W4_BEST_X7 = np.array([0.206363, 0.281987, 0.389442, 0.281544, 0.218827, 0.711599])

next_x7, portal7 = analyse_function_w6(
    func_num     = 7,
    beta_ucb     = 1.5,
    use_ei       = True,
    use_lhs      = True,
    trust_center = W4_BEST_X7,   # anchor to W4 best — not W5 regression
    trust_radius = 0.25,         # expanded from 0.18 — was too tight
    xi           = 0.01,
)


F7 ML Hyperparameters | Dim=6 N=35 BestY=2.6705e+00
Best X* = [0.2064 0.2820 0.3894 0.2815 0.2188 0.7116]
W5 result = 1.6608e+00


  [Grid] Trust-LHS r=0.25 6D 50,000pts


  [GP] Matern(length_scale=0.59, nu=2.5)


  [Sanity] pred=0.9823 actual=0.9823 std=0.001975


  [UCB] b=1.5 max=2.0142 => [0.0219 0.1109 0.1663 0.2760 0.1097 0.7666]
  [EI]  xi=0.01 max=0.216621 => [0.0278 0.2084 0.2708 0.2661 0.1950 0.6987]
  [Ensemble] UCB_mean=0.2557 EI_mean=0.7752 => winner=EI
  [Dist] Distance from best: 0.2288

  >>> SUBMIT F7: 0.027785-0.208441-0.270801-0.266138-0.195016-0.698660 <<<


In [12]:
# F8 — Complex 8D
# W5: 9.9275 — new all-time best. Continue.
W5_BEST_X8 = np.array([0.089787, 0.068251, 0.180968, 0.327284, 0.766207, 0.653365, 0.174832, 0.499246])

next_x8, portal8 = analyse_function_w6(
    func_num     = 8,
    beta_ucb     = 1.5,
    use_ei       = True,
    use_lhs      = True,
    trust_center = W5_BEST_X8,
    trust_radius = 0.30,
    xi           = 0.01,
)


F8 Complex 8D | Dim=8 N=45 BestY=9.9275e+00
Best X* = [0.0898 0.0683 0.1810 0.3273 0.7662 0.6534 0.1748 0.4992]
W5 result = 9.9275e+00
  [Grid] Trust-LHS r=0.30 8D 50,000pts
  [GP] Matern(length_scale=1.29, nu=2.5)
  [Sanity] pred=2.2953 actual=2.2953 std=0.000134


  [UCB] b=1.5 max=2.3451 => [0.0320 0.3443 0.0213 0.0653 0.8290 0.9317 0.0737 0.7502]
  [EI]  xi=0.01 max=0.007680 => [0.0293 0.2853 0.2230 0.0414 0.6254 0.7190 0.0339 0.7974]
  [Ensemble] UCB_mean=2.2755 EI_mean=2.2877 => winner=EI
  [Dist] Distance from best: 0.5169

  >>> SUBMIT F8: 0.029320-0.285338-0.223021-0.041430-0.625367-0.719031-0.033888-0.797446 <<<


In [13]:
print('='*70)
print('WEEK 6 — MODULE 17 — PORTAL SUBMISSION STRINGS')
print('='*70)
portals = {1:portal1, 2:portal2, 3:portal3, 4:portal4,
           5:portal5, 6:portal6, 7:portal7, 8:portal8}
for i in range(1,9):
    print(f'F{i}: {portals[i]}')
print()
print('All-time bests going into W6:')
for i in range(1,9):
    v, w = all_time_best[i]
    print(f'  F{i}: {v:.4e}  ({w})')

WEEK 6 — MODULE 17 — PORTAL SUBMISSION STRINGS
F1: 0.653384-0.652924
F2: 0.704856-0.921380
F3: 0.151659-0.826046-0.622768
F4: 0.291709-0.714786-0.911879-0.664486
F5: 0.102387-0.951309-0.977662-0.978193
F6: 0.394360-0.399099-0.411100-0.595076-0.020599
F7: 0.027785-0.208441-0.270801-0.266138-0.195016-0.698660
F8: 0.029320-0.285338-0.223021-0.041430-0.625367-0.719031-0.033888-0.797446

All-time bests going into W6:
  F1: 2.8209e-04  (W2)
  F2: 6.1120e-01  (Initial)
  F3: -1.0707e-02  (W1)
  F4: -1.2836e-01  (W4)
  F5: 2.9419e+03  (W5)
  F6: -2.9986e-01  (W5)
  F7: 2.6705e+00  (W4)
  F8: 9.9275e+00  (W5)
